# Fine-tuning do Kraken na letra do escrivão

Ajusta o PP-OCRv6 medium (o mesmo modelo do `test_kraken` e do `pipeline_integrado.ipynb`) às 9 páginas de treino. Depois compara antes × depois nas 3 páginas reservadas (7, 10 e 12), que nunca entram no treino. É o mesmo sorteio do `test_kraken`.

**Precisa de GPU:** **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4 serve)**. Em CPU, o treino do medium estoura a memória ou leva horas por época.

**Atenção ao gabarito:** o modelo aprende a grafia do `ground_truth/`, que moderniza palavras e tem erros conhecidos (`ground_truth/comparison_review.md`). Corrija o gabarito antes de um treino definitivo.

Etapas: (1) instalar; (2) gerar as linhas de treino a partir das páginas e do gabarito; (3) treinar; (4) avaliar nas páginas reservadas; (5) opcionalmente salvar no Google Drive.

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = 'feat/pipeline-integrado'  # troque para 'main' depois do merge
REPO = Path('/content/hackathon_ocr_historico')
if Path('/content').exists():
    if not REPO.exists():
        !git clone -q -b {BRANCH} https://github.com/fabiolobato/hackathon_ocr_historico.git {REPO}
    else:  # clone de uma execução anterior (talvez de outro branch): atualiza para BRANCH
        !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only origin {BRANCH}
    os.chdir(REPO)
if not Path('ocr_historico/__init__.py').is_file():
    raise RuntimeError(f'ocr_historico/ não encontrado em {Path.cwd()}. Confira BRANCH ({BRANCH}).')
# O Colab não inclui a pasta atual no sys.path depois do chdir
for pasta in (Path.cwd(), Path.cwd() / 'test_kraken'):
    if str(pasta) not in sys.path:
        sys.path.insert(0, str(pasta))
# instala o repositório no ambiente: os imports sobrevivem a "Reiniciar sessão"
%pip -q install -e . --no-deps
!git log -1 --format='%h %s'

## 1. Dependências e GPU

Se aparecer o aviso de versões antigas carregadas, use **Ambiente de execução → Reiniciar sessão** e continue **a partir desta célula**. O repositório já fica instalado no ambiente pela célula anterior, então os imports funcionam depois do reinício.

In [ ]:
import os
from pathlib import Path
from ocr_historico.dados import RAIZ
from kraken_pipeline.setup import configure_notebook, ensure_dependencies

os.chdir(RAIZ)  # depois de reiniciar a sessão, o Colab volta para /content
configure_notebook()
ensure_dependencies(RAIZ / 'test_kraken' / 'requirements.txt')
%pip -q install jiwer==4.0.0

In [ ]:
import logging
import torch

logging.getLogger('kraken').setLevel(logging.WARNING)
if not torch.cuda.is_available():
    raise RuntimeError('Sem GPU. Ative em Ambiente de execução → Alterar o tipo de ambiente de execução → GPU, '
                       'e execute desde o início.')
DISPOSITIVO = 'cuda:0'
print(torch.cuda.get_device_name(0))

## 2. Linhas de treino

Cada página de treino passa pela mesma entrada da melhor combinação (imagem original ampliada 2×), é segmentada e reconhecida pelo modelo base. O gabarito da página é cortado nas quebras de linha do OCR. Linhas sem correspondência no gabarito (rubricas, números de página) ou mal alinhadas são descartadas e ficam listadas em `finetuning/linhas/manifesto.json`.

In [ ]:
from ocr_historico.dados import listar_paginas
from ocr_historico.finetuning import PASTA_FINETUNING, dividir_paginas, gerar_dataset, treinar
from ocr_historico.motor_kraken import MotorKraken, baixar_modelo

ENTRADA, ESCALA = 'original', 2.0
treino, reserva = dividir_paginas(listar_paginas())
print('treino:', [p.numero for p in treino], '| reservadas:', [p.numero for p in reserva])

# Se o Zenodo estiver fora do ar, coloque o medium.safetensors no seu Drive e informe o caminho
# (ex.: '/content/drive/MyDrive/medium.safetensors'). O MD5 é conferido do mesmo jeito.
MODELO_NO_DRIVE = None
if MODELO_NO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
modelo_base = baixar_modelo('medium', copia_local=Path(MODELO_NO_DRIVE) if MODELO_NO_DRIVE else None)
motor_base = MotorKraken(modelo_base, DISPOSITIVO)
manifesto = gerar_dataset(motor_base, treino, PASTA_FINETUNING / 'linhas', ENTRADA, ESCALA)
print(len(manifesto['usadas']), 'linhas usadas,', len(manifesto['descartadas']), 'descartadas')

In [ ]:
from IPython.display import display
from PIL import Image

for item in manifesto['usadas'][:6]:
    print(f"página {item['pagina']}, linha {item['linha']}\n  OCR base: {item['ocr']}\n  gabarito: {item['gabarito']}")
    display(Image.open(PASTA_FINETUNING / 'linhas' / item['arquivo']))
print('Descartadas (amostra):')
for item in manifesto['descartadas'][:8]:
    print(f"  p{item['pagina']} l{item['linha']}: OCR={item['ocr']!r} gabarito={item['gabarito']!r}")

## 3. Treino

Parte do modelo base e acrescenta ao alfabeto os caracteres que faltarem (`--resize union`). Usa *augmentation* e guarda o melhor checkpoint pela acurácia de validação (10% das linhas, sorteadas). A largura máxima das linhas é calculada a partir do próprio dataset, e o lote 4 com acumulação 4 equivale a um lote de 16 usando cerca de 1/4 da memória. Cada execução grava numa subpasta nova de `finetuning/modelo/`.

In [ ]:
from ocr_historico.finetuning import liberar_gpu

# lote efetivo = LOTE × ACUMULAR = 16. Se ainda faltar memória na GPU: LOTE, ACUMULAR = 2, 8
EPOCAS, LOTE, ACUMULAR, TAXA = 30, 4, 4, 1e-4

# o treino roda em outro processo: tira o modelo base da GPU deste notebook antes
motor_base = None
liberar_gpu()
modelo_treinado = treinar(modelo_base, PASTA_FINETUNING / 'linhas', PASTA_FINETUNING / 'modelo', DISPOSITIVO,
                          epocas=EPOCAS, lote=LOTE, acumular=ACUMULAR, taxa=TAXA)
print('Modelo treinado:', modelo_treinado)

## 4. Antes × depois nas páginas reservadas

Mesma entrada, mesmas páginas, mesmas métricas do `pipeline_integrado.ipynb`.

In [ ]:
from ocr_historico.combinacoes import interpretar
from ocr_historico.executar import avaliar_pasta, executar, imprimir_tabela
from ocr_historico.resultados import carregar, textos_por_arquivo

combinacao = interpretar(f'kraken_{ENTRADA}' + ('_2x' if ESCALA == 2 else ''))
motor_base = MotorKraken(modelo_base, DISPOSITIVO)  # foi tirado da GPU para o treino
motor_ft = MotorKraken(modelo_treinado, DISPOSITIVO)
for rotulo, motor in (('antes', motor_base), ('depois', motor_ft)):
    executar(combinacao, reserva, PASTA_FINETUNING / rotulo, motor_kraken=motor, refazer=True)
    print(f'\n{rotulo.upper()} do fine-tuning (páginas {[p.numero for p in reserva]}):')
    imprimir_tabela(avaliar_pasta(PASTA_FINETUNING / rotulo, reserva))

In [ ]:
pagina = reserva[0]
print('GABARITO\n', pagina.texto_gabarito(), '\n')
for rotulo in ('antes', 'depois'):
    texto = textos_por_arquivo(carregar(PASTA_FINETUNING / rotulo / f'{combinacao.nome}.json'))[pagina.arquivo]
    print(rotulo.upper(), '\n', texto, '\n')

## 5. Salvar no Google Drive (opcional)

O runtime do Colab é apagado ao fechar. Copie o modelo e os resultados para o Drive; o Colab pede autorização ao montar.

In [ ]:
SALVAR_NO_DRIVE = False

if SALVAR_NO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount('/content/drive')
    destino = Path('/content/drive/MyDrive/hackathon_ocr_historico/finetuning')
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copy2(modelo_treinado, destino / modelo_treinado.name)
    for pasta in ('antes', 'depois'):
        shutil.copytree(PASTA_FINETUNING / pasta, destino / pasta, dirs_exist_ok=True)
    shutil.copy2(PASTA_FINETUNING / 'linhas' / 'manifesto.json', destino / 'manifesto.json')
    print('Salvo em', destino)